In [2]:
import pandas as pd

cities = pd.read_csv("../data/cities.csv")

# By organisational form: 200 station based, 67 free floating (including hybrid)

In [95]:
cities["return_to_official_only"].value_counts()

return_to_official_only
True     200
False     67
Name: count, dtype: int64

# By propulsion type: 133 conventional, 38 mixed, 13 electric.

We want to categorize BSSs by propulsion type: a city with 100% of human powered bikes is a conventional BSS, one with 0% human powered bikes is (fully) electric, while one in between is mixed.

With the question concerned, we first want to assign each bike to one primary city, since bikes can have trips in different cities, and we donnot want to count each bike twice in terms of cities. We also exclude bikes with their propulsion type unknown (missing values in bike_types).

In [6]:
bikes = pd.read_csv("../data/bikes.csv")
bike_types = pd.read_csv("../data/bike_types.csv")

bike_types["propulsion_type"].value_counts()

propulsion_type
human              64
electric_assist    27
Name: count, dtype: int64

Step 1: match all bikes with bike_types. Because bike_types specifies the propulsion type.

In [137]:
df = bikes.merge(bike_types, left_on="bike_type_id", right_on="id")
# check number of missing values
print(f"There are {df.shape[0]} bikes, {df["propulsion_type"].isna().sum()} bikes whose propulsion type cannot be identified.")


There are 88436 bikes, 1050 bikes whose propulsion type cannot be identified.


Most of bikes are human powered

In [ ]:
df["propulsion_type"].value_counts()

propulsion_type
human              81530
electric_assist     5856
Name: count, dtype: int64

In [161]:
# This block is used to divide bikes into two subsets based on propulsion types: human powered bikes and electric bikes
bikes_human = df[df["propulsion_type"] == "human"]
bikes_electric = df[df["propulsion_type"] == "electric_assist"]

See Above: there are 88436 bikes, where (81530 + 5856) = 87386 bikes their propulstion types are know, the rest their propulsion types are missing. Only about 1% is missing so we drop them.

Step 2: drop all bikes with unknown propulsion type.

In [ ]:
tmp = bike_types.drop(columns="id")
tmp = tmp.dropna(how="all")

tmp["id"] = bike_types["id"]

bikes_propulsion_type = tmp[["id", "propulsion_type"]].rename(columns={"id": "bike_type_id"})
bikes_propulsion_type.head()

# unfinished here--a mistake. But Job done later part of code. Finish here is more logical.

,bike_type_id,propulsion_type
24,174,human
31,111,human
32,9,human
33,170,human
34,119,electric_assist


Now we have all bikes. we need to know which bikes belong to which city. so we need trips data.

**But first we need to solve this problem with city: a bike should only belong to one city**

Approach: If a bike has trips in multiple cities, it belongs to the city where it has the longest **total trip duration in that city**.

In [151]:
cities = pd.read_csv("../data/cities.csv")
trips = pd.read_csv("../data/trips_small.csv")

print(f"number of cities in cities.csv: {cities["id"].nunique()}")
print(f"number of cities appeared in trips_small.csv: {trips["city_id"].nunique()}")

number of cities in cities.csv: 267
number of cities appeared in trips_small.csv: 185


Interesting discovery: 
1. Among 267 cities only 185 cities have trip records.
2. bikes.csv has 88436 bikes, but trips_small.csv has only trip records of 26424 bikes.


In [152]:
trips.head()
print(f"Interesting discovery: bikes.csv has {bikes["id"].nunique()} bikes, but trips_small.csv has only trip records of {trips["bike_id"].nunique()} bikes.")

Interesting discovery: bikes.csv has 88436 bikes, but trips_small.csv has only trip records of 26424 bikes.


In [153]:
city_bike_duration = (
    trips.groupby(["city_id", "bike_id"])["duration"]
    .sum().
    reset_index(name="total_duration")
)

bike_primary_city = (city_bike_duration.sort_values(by="total_duration", ascending=False)
                     .drop_duplicates(subset="bike_id")
                     [["city_id", "bike_id"]]
                     )

print(f"{city_bike_duration.city_id.nunique()} cities has bike trip,")
print(f"but only {bike_primary_city.city_id.nunique()} primary cities are assigned,\nbecause there is one city where all bikes that appeared in that city had a higher total ride duration in other cities.")

185 cities has bike trip,
but only 184 primary cities are assigned,
because there is one city where all bikes that appeared in that city had a higher total ride duration in other cities.


In [154]:
bike_primary_city.shape[0]
bikes_propulsion_type.head() # id is bike_types.id


,bike_type_id,propulsion_type
24,174,human
31,111,human
32,9,human
33,170,human
34,119,electric_assist


Now we have:
**bike_primary_city**, with bike_id and city_id;
**bikes_propulsion_type**, with bikt_type_id and propulsion_type.
We need to use **bikes** to link them together, to get a df with city_id, bike_id, and propulsion_id

In [155]:
bikes.head()

,id,bike_type_id,computer_id
0,400481,196,83300002927
1,374044,204,7551171765
2,373778,204,7551140444
3,34749,14,0
4,88359,143,7551052355


In [156]:
tmp = bike_primary_city.merge(bikes, left_on="bike_id", right_on="id", how="inner")
city_bike_propulsion = tmp.merge(bikes_propulsion_type, on="bike_type_id")[["city_id", "bike_id", "bike_type_id", "propulsion_type"]]
city_bike_propulsion.head()
city_bike_propulsion.shape

(25833, 4)

Note that all bikes from trips has been assigned with their (primary) cities. But some bikes have NA in propulsion type, while those bikes are dropped in bikes_propulsion_type. By inner merge, these bikes are dropped. The resulting city_bike_propulsion has cities with all bikes whose propulsion types are known.
(25833 < 26424)

Final step: calculate the proportion of human-powered bikes and categorize cities by propulsion types into conventional, mixed, electric.

In [157]:
city_bike_propulsion["city_id"].value_counts()

city_id
391    970
679    894
137    890
497    753
560    704
      ... 
415      9
716      7
741      3
780      1
675      1
Name: count, Length: 184, dtype: int64

In [158]:
tmp = (city_bike_propulsion.assign(is_human=(city_bike_propulsion["propulsion_type"] == "human"))
        .groupby("city_id")["is_human"]
        .mean()
        .reset_index(name="proportion_of_human_powered_bikes")
        )
tmp.shape

(184, 2)

In [159]:
# Categorization
def categorize_propulsion_type(n):
    if n == 1:
        return "conventional"
    elif n == 0:
        return "electric"
    else:
        return "mixed"

tmp["type"] = tmp["proportion_of_human_powered_bikes"].apply(categorize_propulsion_type)
city_propulsion_type = tmp
city_propulsion_type["city_id"].value_counts()

city_id
7      1
23     1
64     1
88     1
128    1
      ..
835    1
848    1
853    1
856    1
858    1
Name: count, Length: 184, dtype: int64

In [160]:
city_propulsion_type["type"].value_counts()


type
conventional    133
mixed            38
electric         13
Name: count, dtype: int64